In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import warnings
warnings.filterwarnings('ignore')

1) 퍼널 분석  
A:본인인증>>B:신분증 인증>>C:개인정보입력>>D:대출 조건 가입>>E:대출 심사>>F:대출 실행

In [3]:
funnel_df = pd.read_csv('/Users/jangjuyeong/Documents/spyder/funnel_data.csv')

In [4]:
agg_funnel = funnel_df.groupby('screen_name').user_id.nunique().reset_index()
agg_funnel.rename(columns={'user_id': 'user_cnt'}, inplace=True)
agg_funnel['conversion'] = agg_funnel['user_cnt'] / agg_funnel.loc[0, 'user_cnt']

In [5]:
agg_funnel

,screen_name,user_cnt,conversion
0,A,10000,1.0000
1,B,8024,0.8024
2,C,2482,0.2482
3,D,2213,0.2213
4,E,1762,0.1762
5,F,1218,0.1218


대부분의 유저가 B단계(신분증 인증화면)에서 이탈했다.

2)문제정의 & 실험  
제휴사가 사용하는 신분증 촬영 & 인식 솔루션에 결함이 있는 것을 확인, 결함 해결을 위한 개발 작업  
실험군과 대조군을 나누어 각각의 그룹에게 다른 화면을 보여주는 실험 진행  
실험결과 분석  
실험군: 화면 A노출유저, 대조군: 기존 화면 노출 유저, 개선지표: C 화면 도달률

In [7]:
abtest_df = pd.read_csv('/Users/jangjuyeong/Documents/abtest_data2.csv')

In [8]:
abtest_df.groupby(['group', 'screen_name']).user_id.nunique()

group  screen_name
A      A              10000
       B               7610
       C               3952
       D               3555
       E               2836
       F               2017
B      A              10000
       B               7914
       C               2735
       D               2424
       E               1945
       F               1363
Name: user_id, dtype: int64

In [9]:
scr_b = abtest_df[abtest_df['screen_name']=='B'][['user_id', 'group']]
scr_c = abtest_df[abtest_df['screen_name']=='C'][['user_id', 'group']]
scr_c['exp_c_yn'] = 1

In [10]:
abtest = pd.merge(scr_b, scr_c, how='left', on=['user_id', 'group'])
abtest.fillna(0, inplace=True)

In [11]:
contingency_table = pd.crosstab(abtest['group'], abtest['exp_c_yn'])

#카이제곱 검정
chi2_stat, p_value_chi2, dof, expected = stats.chi2_contingency(contingency_table)

#결과 출력
print(f"Chi-squared statistic: {chi2_stat}")
print(f"P-value: {p_value_chi2}")
print(f"Degrees of freedom: {dof}")

Chi-squared statistic: 476.7999012015489
P-value: 1.0618980557855836e-105
Degrees of freedom: 1


UX개선이 screen C에 도달하는 비율을 유의미하게 높였다.

In [12]:
conv_a = abtest[abtest['group']=='A'].exp_c_yn.sum() / abtest[abtest['group']=='A'].user_id.nunique()
conv_b = abtest[abtest['group']=='B'].exp_c_yn.sum() / abtest[abtest['group']=='B'].user_id.nunique()

In [13]:
print(f"A안 전환율: {round(conv_a*100, 2)}%")
print(f"B안 전환율: {round(conv_b*100, 2)}%")
print(f"전환율 개선율: {round((conv_a - conv_b) / conv_b * 100,2)}%")

A안 전환율: 51.93%
B안 전환율: 34.56%
전환율 개선율: 50.27%
